<a href="https://colab.research.google.com/github/alyanabilapu/IB9SR0-ASSIGNMENT/blob/main/IB9SR0_Individual_Assignment_5838575.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd

# Data Configuration
DATA_DIR = "/content"
STATIC_TABLES = {
    "branches": "branches.csv",
    "items": "items.csv",
    "members": "members.csv"}

LOANS_HISTORY = {
    "loans_2025_H1": "loans_2025_H1.csv",
    "loans_2025_H2": "loans_2025_H2.csv"} # if in the future, user want to input 2026, 2027, or so on, the files will be input here

In [5]:
def load_bronze_data(DATA_DIR, STATIC_TABLES, LOANS_HISTORY):
    bronze = {}

    # Loading the tables in STATIC_TABLES
    for table_name, filename in STATIC_TABLES.items():
        path = os.path.join(DATA_DIR, filename)
        bronze[table_name] = pd.read_csv(path)

    # Loading the book loan data in LOANS_HISTORY
    for table_name, filename in LOANS_HISTORY.items():
        path = os.path.join(DATA_DIR, filename)
        bronze[table_name] = pd.read_csv(path)

    return bronze

In [6]:
bronze_data = load_bronze_data(DATA_DIR, STATIC_TABLES, LOANS_HISTORY)

# Showing all the data that has been called by the 'load_bronze_data' function, also the number of rows and columns
print("Successfully loaded tables:")
for key in bronze_data.keys():
    print(f"- {key} ({bronze_data[key].shape[0]} rows, {bronze_data[key].shape[1]} columns)")
    print(" ", list(bronze_data[key].columns))

Successfully loaded tables:
- branches (8 rows, 4 columns)
  ['branch_id', 'branch_name', 'area', 'opened_year']
- items (1200 rows, 6 columns)
  ['item_id', 'title', 'item_type', 'category', 'replacement_cost', 'home_branch_id']
- members (832 rows, 6 columns)
  ['member_id', 'join_date', 'age_band', 'membership_type', 'home_branch_id', 'postcode']
- loans_2025_H1 (2815 rows, 8 columns)
  ['loan_id', 'member_id', 'item_id', 'branch_id', 'loan_date', 'due_date', 'return_date', 'fine_charged']
- loans_2025_H2 (3245 rows, 8 columns)
  ['loan_id', 'member_id', 'item_id', 'branch_id', 'return_date', 'due_date', 'loan_date', 'fine_charged']


**Findings:**
1. The columns in both datasets, `loans_2025_H1` and `loans_2025_H2`, are not ordered the same way, which may result in calculation errors downstream. Since both files are in consecutive periods of the same year (1st/2nd half of the year), column ordering should be consistent to enable merging the two files into one dataset.

In [7]:
for key in bronze_data.keys():
    print("=" * 60)
    print(f"Table: {key}")
    print("=" * 60)

    display(bronze_data[key].head(5))
    print("\n")
    print(bronze_data[key].dtypes)
    print("\n")

Table: branches


,branch_id,branch_name,area,opened_year
0,B01,Abbeygate,North,1962
1,B02,Brookfield,North,1974
2,B03,Castlemere,East,1988
3,B04,Dunmore,East,1995
4,B05,Elmstead,South,1971




branch_id      object
branch_name    object
area           object
opened_year     int64
dtype: object


Table: items


,item_id,title,item_type,category,replacement_cost,home_branch_id
0,I0001,The Little Map,Book,Fiction,19.08,B02
1,I0002,The Burning Kingdom,DVD,Film,12.3,B03
2,I0003,The Hidden Forest,DVD,film,17.49,B05
3,I0004,The Broken Engine,Audiobook,Non-fiction,23.59,B01
4,I0005,The Winter Lighthouse,Book,Children,24.48,B07




item_id             object
title               object
item_type           object
category            object
replacement_cost    object
home_branch_id      object
dtype: object


Table: members


,member_id,join_date,age_band,membership_type,home_branch_id,postcode
0,M00001,2015-01-16,35-59,Standard,B02,ZZ8 3GR
1,M00002,2023-01-07,35-59,Standard,B03,ZZ6 1NF
2,M00003,2021-02-23,18-34,Standard,B03,ZZ9 9YT
3,M00004,2016-06-03,60+,Concession,B06,ZZ4 7JG
4,M00005,27/07/2023,Under 12,Junior,B01,zz2 5xy




member_id          object
join_date          object
age_band           object
membership_type    object
home_branch_id     object
postcode           object
dtype: object


Table: loans_2025_H1


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged
0,L000001,M00783,I0479,B01,2025-01-01,2025-01-08,2025-01-08,0.00
1,L000002,M00472,I0474,B07,2025-01-01,2025-01-22,2025-01-22,0.00
2,L000003,M00613,I0458,B07,2025-01-01,2025-01-22,2025-01-09,0.00
3,L000004,M00759,I0344,B08,01/01/2025,15/01/2025,18/01/2025,0.60
4,L000005,M00669,I0450,B01,2025-01-01,2025-01-22,2025-01-26,0.80




loan_id         object
member_id       object
item_id         object
branch_id       object
loan_date       object
due_date        object
return_date     object
fine_charged    object
dtype: object


Table: loans_2025_H2


,loan_id,member_id,item_id,branch_id,return_date,due_date,loan_date,fine_charged
0,L002786,M00494,I0703,B08,2025-07-08,2025-07-22,2025-07-01,0.00
1,L002787,M00788,I0254,B01,2025-07-06,2025-07-22,2025-07-01,0.00
2,L002788,M00514,I0296,B04,14/07/2025,08/07/2025,01/07/2025,0.00
3,L002789,M00177,I0243,B05,2025-07-05,2025-07-22,2025-07-01,0.00
4,L002790,M00454,I0228,B02,2025-07-24,2025-07-22,2025-07-01,0.40




loan_id         object
member_id       object
item_id         object
branch_id       object
return_date     object
due_date        object
loan_date       object
fine_charged    object
dtype: object




**THE DATA**

There are five tables from a fictional community library service dataset. They cover book loans placed for first-half and second-half of the year 2025.

| Name in this notebook | What it holds |
| --- | --- |
| `branches` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `items` | The order's progress through the process, recorded as events |
| `members` | The items inside each order |
| `loans_2025_H1` | Money movements for each order: charges and refunds |
| `loans_2025_H2` | Restaurant details: cuisine and city |

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

---
**BRONZE TO SILVER**

Goal: Clean the data, enforce types, and deduplicate.


In [8]:
def process_silver(bronze_data):
    silver = {}
    for key, df in bronze_data.items():
        silver[key] = df.copy()
    return silver

silver_data = process_silver(bronze_data)

for key in silver_data.keys():
    print(f"Read table: {key} ({silver_data[key].shape[0]} rows, {silver_data[key].shape[1]} columns)")

Read table: branches (8 rows, 4 columns)
Read table: items (1200 rows, 6 columns)
Read table: members (832 rows, 6 columns)
Read table: loans_2025_H1 (2815 rows, 8 columns)
Read table: loans_2025_H2 (3245 rows, 8 columns)


In [9]:
silver_items = silver_data['items'].copy()

unique_item_ids = silver_items['item_id'].nunique()
print(f"Distinct values in column 'item_id': {unique_item_ids}")
print("-" * 50)

unique_item_types = silver_items['item_type'].unique()
unique_categories = silver_items['category'].unique()

print("Unique values in 'item_type':")
print(unique_item_types)
print("\nUnique values in 'category':")
print(unique_categories)
print("-" * 50)

silver_items['replacement_cost_numeric'] = pd.to_numeric(silver_items['replacement_cost'], errors='coerce')

print("Ringkasan Statistik untuk 'replacement_cost':")
display(silver_items['replacement_cost_numeric'].describe())

Distinct values in column 'item_id': 1200
--------------------------------------------------
Unique values in 'item_type':
['Book' 'DVD' 'Audiobook']

Unique values in 'category':
['Fiction' 'Film' 'film' 'Non-fiction' 'Children' 'Reference' 'fiction'
 'Non fiction' 'nonfiction' "Children's" 'reference' 'Fiction '
 'Childrens' 'children' 'FICTION' 'Non-Fiction' 'Films' 'non-fiction '
 'Ref.']
--------------------------------------------------
Ringkasan Statistik untuk 'replacement_cost':


,replacement_cost_numeric
count,1104.000000
mean,15.802283
std,5.557573
min,6.010000
25%,11.260000
50%,15.525000
75%,20.007500
max,29.810000


AI Prompt:

create a code that checking columns 'item_id` how many distinct values, for columns 'item_type' and 'category' list down all unique values, and for replacement_cost check basic summary statistic

Findings:
1. Distinct values in column 'item_id' is 1200 values, and from the first when we load the `items` table has 1200 so it means that there's no duplicated row in the table.
2. For the categorical columns like `item_type` there's 3 types, looks good. And for column 'category' there's so many similar categories with different way of writing, I assume these are inconsistensies in writing, must rework on it later by making a function to standardize categories.
3. In the `replacement_cost` the count shows only 1104 available, so there must be missing values.

Decision to clean, create function of:
1. Check missing values
2. Standardizing typing
3. check any duplicated row/ID

In [16]:
def check_missing_value(df, name="Table"):
    """Hitung missing values (NaN dan string kosong) per kolom."""
    blank_as_nan = df.replace(r"^\s*$", pd.NA, regex=True)
    missing = blank_as_nan.isna().sum()

    report = pd.DataFrame({
        "Total Missing": missing,
        "Percentage (%)": (missing / len(df) * 100).round(2),
    })
    print(f"Missing values: {name} ({len(df):,} rows)")
    display(report)

check_missing_value(silver_data["items"], "items")

# Also check for the other tables
check_missing_value(silver_data["members"], "members")
check_missing_value(silver_data["loans_2025_H1"], "loans_2025_H1")
check_missing_value(silver_data["loans_2025_H2"], "loans_2025_H2")


Missing values: items (1,200 rows)


,Total Missing,Percentage (%)
item_id,0,0.0
title,0,0.0
item_type,0,0.0
category,0,0.0
replacement_cost,24,2.0
home_branch_id,0,0.0


Missing values: members (832 rows)


,Total Missing,Percentage (%)
member_id,0,0.00
join_date,0,0.00
age_band,0,0.00
membership_type,0,0.00
home_branch_id,0,0.00
postcode,33,3.97


Missing values: loans_2025_H1 (2,815 rows)


,Total Missing,Percentage (%)
loan_id,0,0.00
member_id,0,0.00
item_id,0,0.00
branch_id,0,0.00
loan_date,0,0.00
due_date,0,0.00
return_date,42,1.49
fine_charged,286,10.16


Missing values: loans_2025_H2 (3,245 rows)


,Total Missing,Percentage (%)
loan_id,0,0.00
member_id,0,0.00
item_id,0,0.00
branch_id,0,0.00
return_date,230,7.09
due_date,0,0.00
loan_date,0,0.00
fine_charged,488,15.04


AI Prompt:

Create a def check_missing_value function to loop through all columns in items and check if there are any NA or empty cells. Make this function general so it can also be used for other tables.